# Imports & Setup

In [15]:
import os # File handling
import pandas as pd
import numpy as np

In [16]:
current_file_dir = os.getcwd() # Get current directory
project_root = os.path.dirname(current_file_dir) # Get project root

# Ingest

In [17]:
df = pd.read_csv(f"{project_root}/data_raw/frailty_raw.csv") # Load data csv

# Process

In [18]:
df.shape

(10, 5)

In [19]:
df.head()

,Height,Weight,Age,Grip strength,Frailty
0,65.8,112,30,30,N
1,71.5,136,19,31,N
2,69.4,153,45,29,N
3,68.2,142,22,28,Y
4,67.8,144,29,24,Y


In [20]:
df["Weight_kg"] = df["Weight"] * 0.45359237 # Convert Weight
df["Height_m"] = df["Height"] * 0.0254 # Convert Height

In [21]:
df["BMI"] = (df["Weight_kg"] / (df["Height_m"] ** 2)).round(2) # Calc BMI

In [22]:
age_categories = ["<30", "30–45", "46–60", ">60"] # Declare age groups

# Create new feature for categorical age group
df["AgeGroup"] = np.select(
    [
        df["Age"] < 30,
        (df["Age"] >= 30) & (df["Age"] <= 45),
        (df["Age"] >= 46) & (df["Age"] <= 60),
        df["Age"] > 60
    ],
    age_categories,
    default = ">60"
)

**Note for categorical age group one hot encoded features**

This operation did not create a one hot feature for the category ">60" because there are not ages in the data that would fall into that category

In [23]:
# Binary encoding
df["Frailty_binary"] = df["Frailty"].map({
    'N': 0,
    'Y': 1
})

In [24]:
# One hot encoding for age group
df = pd.get_dummies(df,
                    columns=["AgeGroup"],
                    prefix="AgeGroup",
                    dtype=int
                )

In [25]:
df.head()

,Height,Weight,Age,Grip strength,Frailty,Weight_kg,Height_m,BMI,Frailty_binary,AgeGroup_30–45,AgeGroup_46–60,AgeGroup_<30
0,65.8,112,30,30,N,50.802345,1.67132,18.19,0,1,0,0
1,71.5,136,19,31,N,61.688562,1.81610,18.70,0,0,0,1
2,69.4,153,45,29,N,69.399633,1.76276,22.33,0,1,0,0
3,68.2,142,22,28,Y,64.410117,1.73228,21.46,1,0,0,1
4,67.8,144,29,24,Y,65.317301,1.72212,22.02,1,0,0,1


In [26]:
# Save processed data
df.to_csv(f"{project_root}/data_clean/frailty_clean.csv")

# Analyze

In [27]:
# Create summary table df
summary_table_df = pd.DataFrame({
    "mean": df.select_dtypes(include="number").mean(),
    "median": df.select_dtypes(include="number").median(),
    "std": df.select_dtypes(include="number").std()
})

# Convert to Markdown
summary_table_md = summary_table_df.to_markdown()

# Show results
print(summary_table_md)

# Write to file
with open(f"{project_root}/reports/findings.md", "w") as f:
    f.write(summary_table_md)

|                |      mean |    median |        std |
|:---------------|----------:|----------:|-----------:|
| Height         |  68.6     |  68.45    |  1.67066   |
| Weight         | 131.9     | 136       | 14.2318    |
| Age            |  32.5     |  29.5     | 12.8604    |
| Grip strength  |  26       |  27       |  4.52155   |
| Weight_kg      |  59.8288  |  61.6886  |  6.45544   |
| Height_m       |   1.74244 |   1.73863 |  0.0424348 |
| BMI            |  19.682   |  19.185   |  1.78097   |
| Frailty_binary |   0.4     |   0       |  0.516398  |
| AgeGroup_30–45 |   0.3     |   0       |  0.483046  |
| AgeGroup_46–60 |   0.2     |   0       |  0.421637  |
| AgeGroup_<30   |   0.5     |   0.5     |  0.527046  |


In [28]:
# Compute corrolation
corrolation_frailty_grip = df["Grip strength"].corr(df["Frailty_binary"])

# Show results
print("Corrolation grip strength to frailty:", corrolation_frailty_grip)

Corrolation grip strength to frailty: -0.4758668672668007
